In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
import numpy as np
from collections.abc import Callable
from simulators import ModelClass, NestedModelFamily, ContextManager
from simulators.benchmarks import DDM, RDM, CDM

In [5]:
# Custom priors for DDM
ddm_priors = {
    "v": {
        "intercept": lambda: np.random.gamma(2.5, 1.0),  # Drift rate intercept
        "slope": lambda: np.random.normal(0.0, 2.0)      # Drift rate slope
    },
    "s_v": {
        "intercept": lambda: np.random.gamma(1.0, 0.3),  # Drift noise
        "slope": lambda: 0.0                            # Fixed slope
    },
    "a": {
        "intercept": lambda: np.random.gamma(8.0, 0.4),  # Boundary intercept
        "slope": lambda: np.random.normal(0.0, 0.8)      # Boundary slope
    },
    "decay": {
        "intercept": lambda: np.random.gamma(1.0, 0.5),  # Decay rate
        "slope": lambda: 0.0                            # Fixed slope
    },
    "tau": {
        "intercept": lambda: np.random.gamma(2.0, 0.25), # Non-decision time
        "slope": lambda: 0.0                            # Fixed slope
    },
    "s_tau": {
        "intercept": lambda: np.random.uniform(0.0, 0.5), # Non-decision time noise
        "slope": lambda: 0.0                             # Fixed slope
    },
}

# Custom priors for RDM
rdm_priors = {
    "v": {
        "intercept": lambda: np.random.gamma(2.5, 0.9),  # Base drift rate
        "slope": lambda: np.random.normal(0.0, 2.5)      # Drift slope
    },
    "v_diff": {
        "intercept": lambda: np.random.normal(0.0, 1.5), # Differential drift
        "slope": lambda: 0.0                            # Fixed slope
    },
    "a": {
        "intercept": lambda: np.random.gamma(9.0, 0.35), # Boundary intercept
        "slope": lambda: np.random.normal(0.0, 0.9)      # Boundary slope
    },
    "decay": {
        "intercept": lambda: np.random.gamma(1.0, 0.45), # Decay rate
        "slope": lambda: 0.0                            # Fixed slope
    },
    "tau": {
        "intercept": lambda: np.random.gamma(2.5, 0.2),  # Non-decision time
        "slope": lambda: 0.0                            # Fixed slope
    },
}

# Custom priors for CDM
cdm_priors = {
    "v": {
        "intercept": lambda: np.random.normal(1.0, 1.5), # Drift magnitude
        "slope": lambda: np.random.normal(0.0, 1.8)      # Drift slope
    },
    "v_theta": {
        "intercept": lambda: np.random.uniform(-np.pi, np.pi), # Drift angle
        "slope": lambda: 0.0                                   # Fixed slope
    },
    "s_v": {
        "intercept": lambda: np.random.gamma(1.0, 0.25), # Drift noise
        "slope": lambda: 0.0                            # Fixed slope
    },
    "a": {
        "intercept": lambda: np.random.gamma(8.0, 0.4),  # Boundary intercept
        "slope": lambda: np.random.normal(0.0, 0.8)      # Boundary slope
    },
    "decay": {
        "intercept": lambda: np.random.gamma(1.0, 0.5),  # Decay rate
        "slope": lambda: 0.0                            # Fixed slope
    },
    "tau": {
        "intercept": lambda: np.random.gamma(2.0, 0.25), # Non-decision time
        "slope": lambda: 0.0                            # Fixed slope
    },
    "s_tau": {
        "intercept": lambda: np.random.uniform(0.0, 0.5), # Non-decision time noise
        "slope": lambda: 0.0                             # Fixed slope
    },
}

In [6]:
# Initialize models
ddm = DDM()
rdm = RDM()
cdm = CDM()

In [7]:
# Define intrinsic parameters for each model
ddm_params = ["v", "s_v", "a", "decay", "tau", "s_tau"]
rdm_params = ["v", "v_diff", "a", "decay", "tau"]
cdm_params = ["v", "v_theta", "s_v", "a", "decay", "tau", "s_tau"]

In [8]:
# Create context managers
ddm_context_manager = ContextManager(parameter_names=ddm_params)
rdm_context_manager = ContextManager(parameter_names=rdm_params)
cdm_context_manager = ContextManager(parameter_names=cdm_params)

In [13]:
# Create model families
model_families = [
    NestedModelFamily(
        name="DDM",
        model=ddm,
        context_manager=ddm_context_manager,
        prior_fun=ddm_priors,
        intrinsic_params=ddm_params,
    ),
    NestedModelFamily(
        name="RDM",
        model=rdm,
        context_manager=rdm_context_manager,
        prior_fun=rdm_priors,
        intrinsic_params=rdm_params,
    ),
    NestedModelFamily(
        name="CDM",
        model=cdm,
        context_manager=cdm_context_manager,
        prior_fun=cdm_priors,
        intrinsic_params=cdm_params,
    ),
]

In [14]:
model_class = ModelClass(model_families=model_families)

In [15]:
batch_size = 2
samples = model_class.sample(
    model_families=model_families, # Priors are embedded in NestedModelFamily
    batch_size=batch_size,
)

(162,) (162,)
(0,) (0,)
No context provided. Building default context from the model.
# alternatives: 4
No context provided. Building default context from the model.
# alternatives: 4
(15,) (15,)
(15,) (15,)
No context provided. Building default context from the model.
Theta mode: zeros
No context provided. Building default context from the model.
Theta mode: zeros
(168,) (168,)
(21,) (21,)


In [16]:
for model_name, batch in samples.items():
    print(f"\nResults for {model_name}:")
    print(f"  Model Names: {batch['model_names']}")
    print(f"  Number of Observations: {batch['num_obs']}")
    print(f"  Number of Regressors: {batch['num_regressors']}")
    print(f"  Design Matrix Shape: {batch['design_matrices'].shape}")
    print(f"  Parameter Matrix Shape: {batch['param_matrices'].shape}")
    print(f"  Simulation Data Keys: {list(batch['sim_data'].keys())}")
    for key, value in batch['sim_data'].items():
        print(f"    {key} Shape: {value.shape}")


Results for DDM:
  Model Names: ['DDM', 'DDM']
  Number of Observations: [[425.]
 [402.]]
  Number of Regressors: [[9.]
 [0.]]
  Design Matrix Shape: (2, 425, 27)
  Parameter Matrix Shape: (2, 162)
  Simulation Data Keys: ['rts', 'choices']
    rts Shape: (2, 425)
    choices Shape: (2, 425)

Results for RDM:
  Model Names: ['RDM', 'RDM']
  Number of Observations: [[561.]
 [272.]]
  Number of Regressors: [[1.]
 [1.]]
  Design Matrix Shape: (2, 561, 3)
  Parameter Matrix Shape: (2, 15)
  Simulation Data Keys: ['rts', 'choices']
    rts Shape: (2, 561)
    choices Shape: (2, 561)

Results for CDM:
  Model Names: ['CDM', 'CDM']
  Number of Observations: [[117.]
 [ 90.]]
  Number of Regressors: [[8.]
 [1.]]
  Design Matrix Shape: (2, 117, 24)
  Parameter Matrix Shape: (2, 168)
  Simulation Data Keys: ['rts', 'choices']
    rts Shape: (2, 117)
    choices Shape: (2, 117)
